### Step 1: Read the source and target tables

In [0]:
from pyspark.sql import functions as F

incremental_silver_orders = spark.table(
    "workspace.ecommerce_project.incremental_silver_orders"
)

silver_orders = spark.table(
    "workspace.ecommerce_project.silver_orders"
)

### Step 2: Identify new records

In [0]:
new_orders_df = (
    incremental_silver_orders
    .join(
        silver_orders,
        on="order_id",
        how="left_anti"
    )
)

In [0]:
new_orders_df.show()

+---------+-----------+------------------+--------------------+--------------+----------+-------------------+--------+----------+----------------+------------+------------+----------+---------+-------------+--------------+-------+--------------------+---------------+--------+-------------+-----------+--------------+------------+----------------+-----------+-------------+---------------------+----------------------+----------------------+----------------+-------------------+------------------+--------------------+----------------+----------------+------------------+------------------------+--------------------+--------------------+----------------------+----------------------+--------------------+------------------+-----------------+---------------------+-----------------------+--------------------------+--------------------+--------------------+--------------------+--------------------------+---------------------+------------------------------+---------------------+--------------------

In [0]:
print("New records to insert:",new_orders_df.count())

New records to insert: 13


In [0]:
new_orders_df.write.format("delta").mode("append").saveAsTable("workspace.ecommerce_project.silver_orders")

### Step 4: Verify the inserted records

In [0]:
silver_orders_after_insert = spark.table(
    "workspace.ecommerce_project.silver_orders"
)

In [0]:
print("Silver records after INSERT:",silver_orders_after_insert.count())

Silver records after INSERT: 738


In [0]:
duplicate_order_ids = (
    silver_orders_after_insert
    .groupBy("order_id")
    .count()
    .filter(F.col("count")>1)
)

duplicate_order_ids.show()

+--------+-----+
|order_id|count|
+--------+-----+
+--------+-----+



In [0]:
new_orders_after_insert = (
    incremental_silver_orders
    .join(
        silver_orders_after_insert,
        on="order_id",
        how="left_anti"
    )
)

print("New records remaining:",new_orders_after_insert.count())

New records remaining: 0
